# 02 · Encoding: one-hot + padding
Step 2 of the thesis workflow (after `01_jets_meta`).

**Input:** `data/raw/journal_entries_4203.csv` (never modified), `data/derived/meta.parquet` (notebook 01)
**Output:** `data/derived/X_onehot.npz` (sparse, rows in `meta.parquet` order) + `data/derived/onehot_features.json` (column names, feature groups, fitted parameters)

Decisions used here (details and rationale in `results/feature_engineering_plan.md`):

| ID | What | Setting |
|---|---|---|
| D9 | line order inside the padding | debit first, then account number, then amount (largest first) |
| D10 | scalers | fitted on FY1 (2023) only; FY2 values are not clipped |
| D4 | calendar features | `promptly`, `weekend`, `nwh` kept, continuous `lag_days` added |
| D11 | extra time features | `nwh_dist`, hour sin/cos, weekday sin/cos built as switchable groups, **off** in the main setting |

All candidate features are stored once. A model run selects columns by group with `select_columns()`, so no rebuild is needed to switch a group on or off.

In [67]:
# cell_02_01
import sys
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.width", 140)

RAW_REL = Path("data/raw/journal_entries_4203.csv")


def find_project_root(marker: Path = RAW_REL) -> Path:
    # walk up from the working directory until `marker` exists (same as notebook 01)
    here = Path.cwd().resolve()
    for p in (here, *here.parents):
        if (p / marker).exists():
            return p
    raise FileNotFoundError(f"{marker} not found from {here} upwards")


ROOT = find_project_root()
sys.path.insert(0, str(ROOT))
from src.load import load_lines
from src import encode as fe

DATA = ROOT / RAW_REL
DERIVED = ROOT / "data" / "derived"
FY1 = 2023          # tuning year (D6); FY2 = 2024 is the reporting year
print("project root:", ROOT)

project root: C:\MyWorks\Thesis\Jet


## 1. Load raw lines and the meta table

In [54]:
# cell_02_02
lines = load_lines(DATA)                   # src/load.py (adds fy, line_no)
meta = pd.read_parquet(DERIVED / "meta.parquet")

print(lines.shape, meta.shape)
assert len(meta) == 102_429 and lines["posting_id"].nunique() == len(meta)
assert set(lines["posting_id"]) == set(meta["posting_id"])

(256232, 21) (102429, 11)


## 2. Which fields are per line and which are per entry?

If a field has one value inside every `posting_id`, it is encoded **once per entry**, not once per line slot. Repeating it in 4 slots would count it up to 4 times in every distance (LOF, kNN) without adding information.

In [55]:
# cell_02_03
g = lines.groupby("posting_id", sort=False)
check = {c: int((g[c].nunique() > 1).sum())
         for c in ["user", "tax_rate", "entered_date", "document_date", "promptly", "weekend", "nwh",
                   "gl_account", "cd_flag", "amount"]}
pd.Series(check, name="entries with >1 value")

user                  0
tax_rate              0
entered_date          0
document_date         0
promptly              0
weekend               0
nwh                   0
gl_account       102429
cd_flag          102429
amount            51373
Name: entries with >1 value, dtype: int64

In [56]:
# cell_02_04
# expected: 0 for all entry-level fields; > 0 for gl_account, cd_flag, amount (line level)
for c in fe.ENTRY_COLS:
    assert check[c] == 0, c

### Line patterns (debit/credit sequence as written in the CSV)

In [57]:
# cell_02_05
pattern = g["cd_flag"].agg("".join).rename("csv_pattern")
pat = (pd.DataFrame({"csv_pattern": pattern})
       .join(meta.set_index("posting_id")[["label"]])
       .groupby("csv_pattern")["label"].agg(entries="size", anomalies="sum"))
pat

,entries,anomalies
csv_pattern,,
CCD,60,0
DC,51056,542
DCC,18000,91
DDC,33312,217
DDDC,1,0


In [58]:
# cell_02_06
# the only 4-line entry: it alone fills slot 4, so the one-hot encoding will make it very isolated
four = meta.loc[meta["n_lines"] == 4, "posting_id"].tolist()
print(four)
lines[lines["posting_id"].isin(four)].drop(columns=["gl_account_name", "text"])

['p_2']


,posting_id,document_date,posting_date,entered_date,gl_account,amount,cd_flag,user,tax_rate,ref_id,promptly,weekend,nwh,label,marking,top_n,high_cash,fy,line_no
256224,p_2,2024-02-06,2024-02-06,2024-02-06 09:00:00,1700,1490.90,D,Alice,0.19,p_1,1,0,0,0,0,0,0,2024,0
256225,p_2,2024-02-06,2024-02-06,2024-02-06 09:00:00,2020,63.36,D,Alice,0.19,p_1,1,0,0,0,0,0,0,2024,1
256226,p_2,2024-02-06,2024-02-06,2024-02-06 09:00:00,1576,295.31,D,Alice,0.19,p_1,1,0,0,0,0,0,0,2024,2
256227,p_2,2024-02-06,2024-02-06,2024-02-06 09:00:00,1600,1849.57,C,Alice,0.19,p_1,1,0,0,0,0,0,0,2024,3


## 3. Build the encoding

In [59]:
# cell_02_07
X, feature_info, params, sorted_lines, ent = fe.build_onehot(lines, meta, fy1=FY1)
print(X.shape, f"non-zeros: {X.nnz:,}")
pd.Series([f["group"] for f in feature_info]).value_counts().rename("columns per group")

(102429, 140) non-zeros: 1,935,191


lines             129
calendar_flags      5
hour_cyc            2
wday_cyc            2
lag_days            1
nwh_dist            1
Name: columns per group, dtype: int64

## 4. D9 — does the line order matter in this dataset?

Count entries whose CSV line order differs from the D9 order. Each of these would get a different vector than an entry with the same content written in another order.

In [60]:
# cell_02_08
changed = fe.order_changes(sorted_lines)
print(f"entries with CSV order != D9 order: {changed.sum():,} of {len(changed):,} "
      f"({changed.mean():.1%})")
(pd.DataFrame({"changed": changed}).join(pattern)
   .groupby("csv_pattern")["changed"].agg(entries="size", changed="sum"))

entries with CSV order != D9 order: 51,253 of 102,429 (50.0%)


,entries,changed
csv_pattern,,
CCD,60,60
DC,51056,0
DCC,18000,18000
DDC,33312,33192
DDDC,1,1


In [61]:
# cell_02_09
# example for the thesis: one entry before and after sorting
ex = changed[changed].index[0]
cols = ["posting_id", "csv_pos", "slot", "cd_flag", "gl_account", "gl_account_name", "amount"]
sorted_lines.loc[sorted_lines["posting_id"] == ex, cols].sort_values("csv_pos")

,posting_id,csv_pos,slot,cd_flag,gl_account,gl_account_name,amount
0,a_1,0,1,D,3400,"Cost of merchandise, 19% input tax",112.42
1,a_1,1,0,D,1576,"Deductible input tax, 19%",21.36
2,a_1,2,2,C,1600,Trade payables,133.78


In [62]:
# cell_02_10
# tie-break check: same side and same account twice inside one entry?
dup = sorted_lines.groupby(["posting_id", "cd_flag", "gl_account"]).size()
print("entries needing the amount tie-break:", int((dup > 1).sum()))

entries needing the amount tie-break: 0


## 5. Sanity checks

In [63]:
# cell_02_11
names = [f["name"] for f in feature_info]
col = {n: i for i, n in enumerate(names)}
is1 = (meta["fy"] == FY1).to_numpy()

def colv(name):
    return np.asarray(X[:, col[name]].todense()).ravel()

# (a) every slot has exactly one account category and one side category (PAD included)
for s in range(1, fe.MAX_LINES + 1):
    a = [i for i, f in enumerate(feature_info) if f["slot"] == s and f["name"].startswith(f"s{s}_acct_")]
    c = [i for i, f in enumerate(feature_info) if f["slot"] == s and f["name"].startswith(f"s{s}_cd_")]
    assert (np.asarray(X[:, a].sum(axis=1)).ravel() == 1).all()
    assert (np.asarray(X[:, c].sum(axis=1)).ravel() == 1).all()

# (b) slot use matches n_lines
for s in range(1, fe.MAX_LINES + 1):
    filled = colv(f"s{s}_acct_PAD") == 0
    assert (filled == (meta["n_lines"].to_numpy() >= s)).all()

# (c) scaled values: FY1 in [0, 1], FY2 may leave the range (not clipped, D10)
for nm in ["s1_amount", "lag_days", "nwh_dist"]:
    v = colv(nm)
    print(f"{nm:10s} FY1 [{v[is1].min():.3f}, {v[is1].max():.3f}]   FY2 [{v[~is1].min():.3f}, {v[~is1].max():.3f}]")
    assert v[is1].min() >= -1e-9 and v[is1].max() <= 1 + 1e-9

# (d) nwh_dist = 0 exactly when nwh = 0
assert ((ent["nwh_dist_raw"].to_numpy() > 0) == (ent["nwh"].to_numpy() == 1)).all()

# (e) promptly bins agree with lag_days (1 = 0-9, 2 = 10-29, 3 = 30+ days)
bins = pd.cut(ent["lag_days_raw"], [-np.inf, 9, 29, np.inf], labels=[1, 2, 3]).astype(int)
print("promptly vs lag bins, mismatches:", int((bins.to_numpy() != ent["promptly"].to_numpy()).sum()))

# (f) weekend flag agrees with the weekday of entered_date (Sat = 5, Sun = 6)
wk = np.select([ent["wday"] == 5, ent["wday"] == 6], [1, 2], 0)
print("weekend vs weekday, mismatches:", int((wk != ent["weekend"].to_numpy()).sum()))

s1_amount  FY1 [0.016, 1.000]   FY2 [0.016, 1.000]
lag_days   FY1 [0.000, 1.000]   FY2 [0.000, 0.802]
nwh_dist   FY1 [0.000, 1.000]   FY2 [0.000, 1.052]
promptly vs lag bins, mismatches: 0
weekend vs weekday, mismatches: 0


## 6. Save

In [64]:
# cell_02_12
fe.save_encoding(X, feature_info, params,
                 DERIVED / "X_onehot.npz", DERIVED / "onehot_features.json")
print("saved:", DERIVED / "X_onehot.npz", DERIVED / "onehot_features.json")

saved: C:\MyWorks\Thesis\Jet\data\derived\X_onehot.npz C:\MyWorks\Thesis\Jet\data\derived\onehot_features.json


## 7. How later notebooks use it

Load once, then choose the feature groups per run. Write `fe.groups_label(groups)` into the `feature_groups` column of `results/runs.csv`.

In [65]:
# cell_02_13
X, feature_info, params = fe.load_encoding(DERIVED / "X_onehot.npz", DERIVED / "onehot_features.json")

main = fe.FEATURE_GROUPS_MAIN
X_main, names_main = fe.select_columns(X, feature_info, main)
print(fe.groups_label(main), X_main.shape)

all_on = {g: True for g in fe.ALL_GROUPS}
X_all, _ = fe.select_columns(X, feature_info, all_on)
print(fe.groups_label(all_on), X_all.shape)

proposal_variant = {**main, "calendar_flags": False, "lag_days": False}
print(fe.groups_label(proposal_variant), fe.select_columns(X, feature_info, proposal_variant)[0].shape)

lines+calendar_flags+lag_days (102429, 135)
lines+calendar_flags+lag_days+nwh_dist+hour_cyc+wday_cyc (102429, 140)
lines (102429, 129)


## Notes for the thesis (Ch. 3, feature engineering)

- Fill in the numbers from sections 2, 4 and 5 (entries with changed line order, the single 4-line entry, mismatch counts).
- `user` and `tax_rate` are constant inside every entry, so they are encoded once per entry, not per line slot.
- One entry has 4 lines; it alone uses slot 4. Check later where it ranks in the one-hot runs: if it appears at the top, it is a false positive caused by the encoding (the entry is normal), worth one sentence.
- Account / user / tax categories are taken from the whole ledger (no labels involved); all numeric scalers are fitted on FY1 only (D10).